In [0]:
SELECT *
FROM detection.tv_input_stats_firehose
WHERE next_create_timestamp >= CURRENT_DATE
AND total_duration > 0
AND fk_input_source_id = 56
-- AND input_device = 'XFINITY'
AND input_device_type
ORDER BY fk_tvid
LIMIT 100

In [0]:
  SELECT * FROM detection.viewing_content_firehose vc
  WHERE fk_tvid = 3317193
  AND vc.session_start >= '2025-02-17'
  AND vc.session_start < '2025-02-24'
  AND vc.fk_zoo_id = 17
  AND vc.fk_input_source_id IS NOT NULL
  AND vc.fk_input_source_id = 56

In [0]:
SELECT * FROM detection.input_source --WHERE input_source_id = 56

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_0;
CREATE TABLE dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_0 AS
WITH vcf AS (
  SELECT * FROM detection.viewing_content_firehose vc
  WHERE vc.session_start >= '2025-02-17T00:00:00'::TIMESTAMP
  AND vc.session_start < '2025-02-24T00:00:00'::TIMESTAMP
  AND vc.fk_zoo_id = 17
  AND vc.fk_input_source_id IS NOT NULL
)
, ld AS (
  SELECT * FROM detection.logo_detection ld
  WHERE ld.next_match_ts >= '2025-02-17T00:00:00'::TIMESTAMP
    -- AND ld.tvid = 3317193
)
, tis AS (
  SELECT * FROM detection.tv_inputsource tis
  WHERE tis.next_create_timestamp >= '2025-02-17T00:00:00'::TIMESTAMP
    -- AND tis.fk_tvid = 3317193
)
, ip AS (
  SELECT * FROM detection.tv_ip_address AS ip
  WHERE ip.next_create_timestamp >= '2025-02-17T00:00:00'::TIMESTAMP
  -- AND ip.fk_tvid = 3317193
)
, st AS (
  SELECT st.*
  FROM detection.epg_station st
  JOIN detection.inscape_station_map ism
    ON ism.mapped_vendor_station_id = st.station_id
   AND ism.mapped_vendor = st.vendor_name
  WHERE st.attributed = 'TRUE'
     OR st.ingested = 'TRUE'
)
SELECT vc.fk_tvid
, vc.fk_input_source_id
, DATE_ADD(DATE_TRUNC('week', vc.session_start), 7) AS next_week_start -- Add 1 week
, ip.isp
, COALESCE(lookup.title, hi.cleaned_hdmi_input_name) AS input_device_prelim 
, MAX(CASE WHEN lookup.title IS NOT NULL THEN 'logo detection' when hi.cleaned_hdmi_input_name IS NOT NULL THEN 'cec' end) AS logo_cec
, SUM(session_duration) AS total_duration_by_input_source_isp_input_device
, SUM(CASE WHEN coalesce(fk_show_id,tms_show_id,tuner_channel_id,tms_tuner_channel_id) IS NOT NULL THEN session_duration ELSE 0 END) AS detected_duration_by_input_source 
, SUM(CASE WHEN coalesce(s.local_or_national, tms_st.local_or_national) = 'Local' THEN session_duration ELSE 0 END) as detected_local_duration_by_input_source 
, SUM(SUM(CASE WHEN frame_category = 'HD' THEN session_duration ELSE 0 END)) OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id) AS hd_frames_duration_by_input_source
, SUM(SUM(CASE WHEN frame_category = 'SD' THEN session_duration ELSE 0 END)) OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id) AS sd_frames_duration_by_input_source
, SUM(SUM(CASE WHEN frame_category = 'OTHER' THEN session_duration ELSE 0 END)) OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id) AS other_frames_duration_by_input_source
, SUM(SUM(session_duration)) OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id) AS total_duration_by_input_source
FROM vcf AS vc
LEFT JOIN detection.frames_stats_firehose
  ON frames_stats_firehose.frame_id = vc.fk_frame_id
LEFT OUTER JOIN st AS s
  ON COALESCE(vc.fk_station_id, vc.tuner_channel_id) = s.station_id
  AND s.vendor_name = 'TIVO'
LEFT OUTER JOIN st AS tms_st
  ON COALESCE(vc.fk_station_id, vc.tuner_channel_id) IS NULL
  AND tms_st.station_id = COALESCE(vc.tms_station_id, vc.tms_tuner_channel_id)
  AND tms_st.vendor_name = 'TMS'
LEFT OUTER JOIN ip
  ON vc.session_start >= ip.create_timestamp
  AND vc.session_start < ip.next_create_timestamp
  AND vc.fk_tvid = ip.fk_tvid
LEFT OUTER JOIN tis
  ON vc.session_start >= tis.create_timestamp
  AND vc.session_start < tis.next_create_timestamp
  AND vc.fk_tvid = tis.fk_tvid
  AND vc.fk_input_source_id = tis.fk_input_source_id
LEFT OUTER JOIN detection.hdmi_input hi
  ON tis.fk_hdmi_input_id = hi.hdmi_input_id
LEFT OUTER JOIN ld
  ON vc.session_start >= ld.view_ts
  AND vc.session_start < ld.next_match_ts
  AND vc.fk_tvid = ld.tvid
  AND vc.fk_input_source_id = ld.fk_input_source_id
LEFT OUTER JOIN detection.logo_detection_lookup lookup
  ON ld.corrected_logo_id = lookup.id
  AND upper(lookup.category) IN ('MVPD', 'OTT DEVICE','GAMING')
GROUP BY 1, 2, 3, 4, 5

In [0]:
SELECT CASE WHEN i.type LIKE '%DMI%' THEN 'HDMI'
            WHEN i.type LIKE 'CAST%' OR i.type IN ('NATIVE APP', 'VIA PLUS') THEN 'APP'
            WHEN i.type IN ('AV', 'COMPONENT', 'COMPOSITE') THEN 'AV'
            WHEN i.type IN ('DIGITAL ANTENNA', 'USB', 'TV', 'ANALOG TV') THEN i.type
            WHEN i.type IS NULL THEN NULL
            ELSE 'OTHER' END AS type
, i.input_source_id, SUM(total_duration_by_input_source_isp_input_device)/3600.0 AS ttl_durt
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_4 s0
JOIN detection.input_source i ON s0.fk_input_source_id = i.input_source_id
GROUP BY 1, 2

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_1;
CREATE TABLE dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_1 AS
SELECT *
, ROW_NUMBER() OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id ORDER BY CASE WHEN input_device_prelim IS NULL THEN 1 ELSE 0 END
, CASE WHEN isp IS NULL THEN 1 ELSE 0 END
, total_duration_by_input_source_isp_input_device DESC) AS rnum_final
-- percentages by input source only
, CASE WHEN total_duration_by_input_source > 0 THEN 1.0 * detected_duration_by_input_source / total_duration_by_input_source ELSE NULL END AS detection_rate
, CASE WHEN total_duration_by_input_source > 0 THEN 1.0 * hd_frames_duration_by_input_source / total_duration_by_input_source ELSE NULL END AS percent_hd_frames
, CASE WHEN total_duration_by_input_source > 0 THEN 1.0 * sd_frames_duration_by_input_source / total_duration_by_input_source ELSE NULL END AS percent_sd_frames
, CASE WHEN total_duration_by_input_source > 0 THEN 1.0 * other_frames_duration_by_input_source / total_duration_by_input_source ELSE NULL END AS percent_other_frames
, CASE WHEN detected_duration_by_input_source > 0 THEN round(1.0 * detected_local_duration_by_input_source / detected_duration_by_input_source,2) ELSE NULL END AS percent_local_frames -- changed this 1/25 to 2 decimals.
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_0 AS vc

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_2;
CREATE TABLE dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_2 AS
SELECT s1.*
, CASE WHEN input_source.type IN ('CAST','NATIVE APP','VIAPLUS') THEN 'APPS'
       WHEN input_source.type = 'USB' THEN 'OTHER'
       WHEN input_source.type = 'ANALOG TV' THEN 'SD TV'
       WHEN input_source.type = 'DIGITAL ANTENNA' AND percent_hd_frames > 0.85 THEN 'HD TV'
       WHEN input_source.type = 'DIGITAL ANTENNA'
         AND percent_hd_frames > percent_sd_frames
         AND percent_hd_frames > percent_other_frames
         AND percent_local_frames >= 0.99 THEN 'HD TV'
       WHEN input_source.type = 'DIGITAL ANTENNA'
         AND percent_sd_frames > percent_hd_frames
         AND percent_sd_frames > percent_other_frames 
         AND percent_local_frames >=0.99 THEN 'SD TV'
       WHEN input_source.type = 'DIGITAL ANTENNA' 
         AND percent_other_frames > percent_hd_frames
         AND percent_other_frames > percent_sd_frames 
         AND percent_local_frames >=0.99 THEN 'OTHER'
       WHEN detection_rate > 0.2 AND percent_hd_frames > 0.85 THEN 'HD TV'
       WHEN detection_rate > 0 
        AND detection_rate <= 0.2 AND percent_hd_frames > percent_sd_frames
        AND percent_hd_frames > percent_other_frames
        AND input_device_prelim IN ('XFINITY','DirecTV','Spectrum','Dish','Verizon_Fios','Cox','Altice',
       'Uverse','MediaCom','Frontier','TiVo','Cable One'
       ) THEN 'HD TV'
       WHEN detection_rate > 0 AND detection_rate <= 0.2 
        AND percent_hd_frames > percent_sd_frames
        AND percent_hd_frames > percent_other_frames
        AND (input_device_prelim NOT IN ('XFINITY','DirecTV','Spectrum','Dish','Verizon_Fios','Cox','Altice', 'Uverse','MediaCom','Frontier','TiVo','Cable One') OR input_device_prelim IS NULL) THEN CASE WHEN input_source.type = 'DIGITAL ANTENNA' THEN 'OTHER' ELSE 'OTT' END
       WHEN percent_sd_frames > percent_hd_frames AND percent_sd_frames > percent_other_frames THEN 'SD TV'
       ELSE 'OTHER'
  END AS category
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_1 as s1
JOIN detection.input_source ON input_source.input_source_id = s1.fk_input_source_id

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_3;
CREATE TABLE dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_3 AS
SELECT s2.*
, CASE WHEN input_source.type IN ('CAST', 'NATIVE APP', 'VIAPLUS','USB') THEN input_source.type
       WHEN input_source.type IN ('ANALOG TV','DIGITAL ANTENNA')
        AND percent_local_frames >= .99
        AND category IN ('SD TV','HD TV', 'OTHER') THEN 'OTA'
       WHEN input_source.type = 'ANALOG TV' THEN input_source.type
       WHEN input_source.type = 'DIGITAL ANTENNA'
        AND percent_hd_frames > 0.85 THEN 'HD DTV'
       WHEN input_source.type = 'DIGITAL ANTENNA'
        AND percent_sd_frames > percent_hd_frames 
        AND percent_sd_frames > percent_other_frames THEN 'SD DTV'
       WHEN detection_rate > 0.2
        AND percent_hd_frames > 0.85 THEN CASE WHEN input_source.type = 'DIGITAL ANTENNA' THEN 'HD DTV' ELSE 'HD STB' END
       WHEN detection_rate > 0
        AND detection_rate <= 0.2
        AND percent_hd_frames > percent_sd_frames
        AND percent_hd_frames > percent_other_frames
        AND input_device_prelim IN (
          'XFINITY','DirecTV','Spectrum','Dish','Verizon_Fios','Cox','Altice', 'Uverse','MediaCom','Frontier','TiVo','Cable One'
          ) THEN CASE WHEN input_source.type = 'DIGITAL ANTENNA' THEN 'HD DTV' ELSE 'HD STB' END
       WHEN detection_rate > 0
        AND detection_rate <= 0.2
        AND percent_hd_frames > percent_sd_frames
        AND percent_hd_frames > percent_other_frames THEN CASE WHEN input_source.type = 'DIGITAL ANTENNA' THEN 'OTHER' ELSE 'OTT' END
       WHEN detection_rate = 0 AND percent_hd_frames > percent_sd_frames AND percent_hd_frames > percent_other_frames THEN 'UNINGESTED'
       WHEN detection_rate > 0.2 AND percent_sd_frames > percent_hd_frames AND percent_sd_frames > percent_other_frames THEN 'SD STB'
       ELSE 'OTHER'
  END AS subcategory
, input_source.type AS input_source_type
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_2 s2
JOIN detection.input_source ON input_source.input_source_id = s2.fk_input_source_id

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_4;
CREATE TABLE dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_4 AS
SELECT *
, CASE WHEN category = 'APPS' THEN NULL
       WHEN subcategory = 'OTA' THEN subcategory
       WHEN category = 'SD TV'
        AND input_device_prelim IN ('Amazon_Fire','Chromecast','Roku','Apple_TV','PlayStation 4','Nintendo Switch','PlayStation 5','XBOX') THEN NULL
       WHEN category = 'HD TV'
        AND subcategory = 'HD DTV' 
        AND input_device_prelim  IN ('Amazon_Fire','Chromecast','Roku','Apple_TV','PlayStation 4','Nintendo Switch','PlayStation 5','XBOX' )  THEN NULL
       WHEN category = 'OTHER' AND subcategory = 'USB' THEN NULL
       WHEN input_source_type = 'DIGITAL ANTENNA' THEN NULL
       WHEN input_device_prelim IS NOT NULL THEN input_device_prelim
       WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND isp IN ('Spectrum','Time%20Warner%20Cable','Charter%20Communications') AND detection_rate>=.2  THEN 'Spectrum'
       WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND isp IN ('Comcast%20Cable') AND detection_rate>=.2 THEN 'XFINITY'
       WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND isp IN ('Verizon%20Fios','Verizon%20Internet%20Services','Verizon%20Wireless') AND detection_rate>=.2 THEN 'Verizon_Fios'
       WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND isp IN ('Cox%20Communications') AND detection_rate>=.2 THEN 'Cox'
       WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND isp IN ('Suddenlink%20Communications','Optimum%20Online') AND detection_rate>=.2 THEN 'Altice'
       WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND isp IN ('Mediacom%20Cable') AND detection_rate>=.2 THEN 'MediaCom'
       WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND isp IN ('Cable%20One') AND detection_rate>=.2 THEN 'Cable One'
       ELSE NULL
END AS input_device
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_3 as s3;

In [0]:
DROP TABLE IF EXISTS dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_5;
CREATE TABLE dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_5 AS
SELECT fk_tvid
, fk_input_source_id
, ROW_NUMBER() OVER (PARTITION BY fk_tvid, next_week_start ORDER BY detected_duration_by_input_source DESC) AS input_number
, category
, subcategory
, CAST(detection_rate AS DECIMAL(4,2)) AS detection_rate
, CAST(percent_hd_frames AS DECIMAL(4,2)) AS percent_hd_frames
, CAST(percent_sd_frames AS DECIMAL(4,2)) AS percent_sd_frames
, CAST(percent_other_frames AS DECIMAL(4,2)) AS percent_other_frames        
, total_duration_by_input_source AS total_duration
, next_week_start AS create_timestamp
, '2100-01-01 00:00:00'::timestamp AS next_create_timestamp
, percent_local_frames
, input_device
, CASE WHEN input_device IS NULL OR input_device = 'OTA' OR input_source_type = 'DIGITAL ANTENNA' THEN NULL
       WHEN logo_cec IS NOT NULL THEN logo_cec
       WHEN category IN ('SD TV','HD TV') AND detection_rate>=.2 
        AND isp IN ('Spectrum','Time%20Warner%20Cable','Charter%20Communications','Comcast%20Cable',
                    'Verizon%20Fios','Verizon%20Internet%20Services','Verizon%20Wireless',
                    'Cox%20Communications','Suddenlink%20Communications','Optimum%20Online',
                    'Mediacom%20Cable','Cable%20One') THEN 'isp'
      ELSE NULL
  END AS input_device_source
, CASE WHEN input_device IN ('PlayStation 4','Nintendo Switch','PlayStation 5','XBOX') THEN 'Gaming'
        WHEN input_device IN ('Amazon_Fire','Chromecast','Roku','Apple_TV') THEN 'OTT Device'
        WHEN input_device IN ('XFINITY','DirecTV','Spectrum','Dish','Verizon_Fios','Cox','Altice','Uverse','MediaCom','Frontier','TiVo','Cable One' ) THEN 'MVPD'
        ELSE NULL
  END AS input_device_type
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_4 s4
WHERE rnum_final = 1;

In [0]:
SELECT CASE WHEN i.type = 'DIGITAL ANTENNA' THEN 'DTV' Else 'Not DTV' END AS dtv_or_not
, COALESCE(category, 'None') AS category
, COALESCE(input_device, 'None') AS input_device
, COALESCE(input_device_type, 'None') AS input_device_type
, COALESCE(subcategory, 'None') AS subcategory
, COALESCE(input_device_source, 'None') AS input_device_source
, COUNT(DISTINCT fk_tvid) AS tv_count
, SUM(total_duration)/3600.0 AS total_duration
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_5 s5
JOIN detection.input_source i ON i.input_source_id = s5.fk_input_source_id
WHERE next_create_timestamp >= CURRENT_DATE
  AND total_duration > 0
  -- AND fk_input_source_id = 56
GROUP BY 1, 2, 3, 4, 5, 6

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT CASE WHEN i.type = 'ANALOG TV' THEN 'DTV' Else 'Not ATV' END AS dtv_or_not
, COALESCE(category, 'None') AS category
, COALESCE(input_device, 'None') AS input_device
, COALESCE(input_device_type, 'None') AS input_device_type
, COALESCE(subcategory, 'None') AS subcategory
, COALESCE(input_device_source, 'None') AS input_device_source
, COUNT(DISTINCT fk_tvid) AS tv_count
, SUM(total_duration)/3600.0 AS total_duration
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_5 s5
JOIN detection.input_source i ON i.input_source_id = s5.fk_input_source_id
WHERE next_create_timestamp >= CURRENT_DATE
  AND total_duration > 0
  -- AND fk_input_source_id = 56
GROUP BY 1, 2, 3, 4, 5, 6

In [0]:
SELECT CASE WHEN i.type LIKE '%DMI%' THEN 'HDMI'
            WHEN i.type LIKE 'CAST%' OR i.type IN ('NATIVE APP', 'VIA PLUS') THEN 'APP'
            WHEN i.type IN ('AV', 'COMPONENT', 'COMPOSITE') THEN 'AV'
            WHEN i.type IN ('DIGITAL ANTENNA', 'USB', 'TV', 'ANALOG TV') THEN i.type
            WHEN i.type IS NULL THEN NULL
            ELSE 'OTHER' END AS type
, COALESCE(category, 'None') AS category
, COALESCE(input_device, 'None') AS input_device
, COALESCE(input_device_type, 'None') AS input_device_type
, COALESCE(subcategory, 'None') AS subcategory
, COALESCE(input_device_source, 'None') AS input_device_source
, COUNT(*)
, COUNT(DISTINCT fk_tvid)
, SUM(total_duration)/3600.0
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_5 s5
JOIN detection.input_source i ON i.input_source_id = s5.fk_input_source_id
WHERE next_create_timestamp >= CURRENT_DATE
  AND total_duration > 0
  -- AND fk_input_source_id = 56
GROUP BY 1, 2, 3, 4, 5, 6

Databricks visualization. Run in Databricks to view.

In [0]:
SELECT CASE WHEN fk_input_source_id = 56 THEN 'DTV' Else 'Not DTV' END AS dtv_or_not
, COALESCE(category, 'None') AS category
, COALESCE(input_device, 'None') AS input_device
, COALESCE(input_device_type, 'None') AS input_device_type
, COALESCE(subcategory, 'None') AS subcategory
, COALESCE(input_device_source, 'None') AS input_device_source
, COUNT(*)
, COUNT(DISTINCT fk_tvid)
, SUM(total_duration)/3600.0
FROM dev.mohit_gangwani.validation_test_for_input_stats_dtv_mvpd_issue_step_5
WHERE next_create_timestamp >= CURRENT_DATE
  AND total_duration > 0
  -- AND fk_input_source_id = 56
GROUP BY 1, 2, 3, 4, 5, 6

In [0]:
%python
def controllerFunction(staging_df):

    deltaHelpers.removeAllTempTablesForSession()
    deltaHelpers.saveToDeltaTempTable(staging_df, "staging_df")

    input_stats_0 = spark.sql(f"""
            SELECT vc.fk_tvid
                        , vc.fk_input_source_id
                        , DATE_ADD(DATE_TRUNC('week', session_start), 7) AS next_week_start -- Add 1 week
                        , ip.isp
                        , COALESCE(lookup.title, hi.cleaned_hdmi_input_name) AS input_device_prelim 
                        , MAX(CASE WHEN lookup.title IS NOT NULL THEN 'logo detection' when hi.cleaned_hdmi_input_name IS NOT NULL THEN 'cec' end) AS logo_cec
                        , SUM(session_duration) AS total_duration_by_input_source_isp_input_device
                        , SUM(CASE WHEN coalesce(fk_show_id,tms_show_id,tuner_channel_id,tms_tuner_channel_id) IS NOT NULL THEN session_duration ELSE 0 END) AS detected_duration_by_input_source 
                        , SUM(CASE WHEN coalesce(s.local_or_national, tms_st.local_or_national) = 'Local' THEN session_duration ELSE 0 END) as detected_local_duration_by_input_source 
                        , SUM(SUM(CASE WHEN frame_category = 'HD' THEN session_duration ELSE 0 END)) OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id) AS hd_frames_duration_by_input_source
                        , SUM(SUM(CASE WHEN frame_category = 'SD' THEN session_duration ELSE 0 END)) OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id) AS sd_frames_duration_by_input_source
                        , SUM(SUM(CASE WHEN frame_category = 'OTHER' THEN session_duration ELSE 0 END)) OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id) AS other_frames_duration_by_input_source
                        , SUM(SUM(session_duration)) OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id) AS total_duration_by_input_source
                FROM {catalog_temp}.{db_name_temp}.staging_df vc 
                LEFT JOIN detection.frames_stats_firehose
                    ON frames_stats_firehose.frame_id = vc.fk_frame_id   
--                LEFT OUTER JOIN detection.inscape_station_map AS station_map
--                    ON station_map.inscape_station_id = coalesce(vc.fk_station_id, vc.tuner_channel_id)
--                    AND station_map.mapped_vendor = 'TMS'    
                LEFT OUTER JOIN detection.epg_station AS s
                    ON COALESCE(vc.fk_station_id, vc.tuner_channel_id) = s.station_id
                    AND s.vendor_name = 'TIVO'
                LEFT OUTER JOIN detection.epg_station tms_st
                  ON COALESCE(vc.fk_station_id, vc.tuner_channel_id) IS NULL
                 AND tms_st.station_id = COALESCE(vc.tms_station_id, vc.tms_tuner_channel_id)
                 AND tms_st.vendor_name = 'TMS'
                LEFT OUTER JOIN detection.tv_ip_address AS ip
                    ON vc.session_start >= ip.create_timestamp
                    AND vc.session_start < ip.next_create_timestamp
                    AND vc.fk_tvid = ip.fk_tvid
                LEFT OUTER JOIN detection.tv_inputsource tis
                    ON vc.session_start >= tis.create_timestamp
                    AND vc.session_start < tis.next_create_timestamp
                    AND vc.fk_tvid = tis.fk_tvid
                    AND vc.fk_input_source_id = tis.fk_input_source_id
                LEFT OUTER JOIN detection.hdmi_input hi
                    ON tis.fk_hdmi_input_id = hi.hdmi_input_id
                LEFT OUTER JOIN detection.logo_detection ld
                    ON vc.session_start >= ld.view_ts
                    AND vc.session_start < ld.next_match_ts
                    AND vc.fk_tvid = ld.tvid
                    AND vc.fk_input_source_id = ld.fk_input_source_id
                LEFT OUTER JOIN detection.logo_detection_lookup lookup
                    ON ld.corrected_logo_id = lookup.id
                    AND upper(lookup.category) IN ('MVPD', 'OTT DEVICE','GAMING')
               -- WHERE session_start >= '{from_date}'::date AND session_start < '{to_date}'::date
                    AND vc.fk_zoo_id = 17
                    AND vc.fk_input_source_id IS NOT NULL
                GROUP BY 1, 2, 3, 4, 5          
            """)
    
    input_stats_0 = deltaHelpers.saveToDeltaTempTable(input_stats_0, "input_stats_0")

    input_stats_1 = spark.sql(f"""
            SELECT *
                , ROW_NUMBER() OVER (PARTITION BY vc.fk_tvid, vc.fk_input_source_id ORDER BY CASE WHEN input_device_prelim IS NULL THEN 1 ELSE 0 END
                , CASE WHEN isp IS NULL THEN 1 ELSE 0 END
                , total_duration_by_input_source_isp_input_device DESC) AS rnum_final
             -- percentages by input source only
                 , CASE WHEN total_duration_by_input_source > 0 THEN 1.0 * detected_duration_by_input_source / total_duration_by_input_source ELSE NULL END AS detection_rate
                 , CASE WHEN total_duration_by_input_source > 0 THEN 1.0 * hd_frames_duration_by_input_source / total_duration_by_input_source ELSE NULL END AS percent_hd_frames
                 , CASE WHEN total_duration_by_input_source > 0 THEN 1.0 * sd_frames_duration_by_input_source / total_duration_by_input_source ELSE NULL END AS percent_sd_frames
                 , CASE WHEN total_duration_by_input_source > 0 THEN 1.0 * other_frames_duration_by_input_source / total_duration_by_input_source ELSE NULL END AS percent_other_frames
                 , CASE WHEN detected_duration_by_input_source > 0 THEN round(1.0 * detected_local_duration_by_input_source / detected_duration_by_input_source,2) ELSE NULL END AS percent_local_frames -- changed this 1/25 to 2 decimals.  
                 FROM {catalog_temp}.{db_name_temp}.input_stats_0 AS vc
                
                """)
    input_stats_1 = deltaHelpers.saveToDeltaTempTable(input_stats_1, "input_stats_1")


    input_stats_2 = spark.sql(f""" 
                SELECT s1.*, 
                      CASE WHEN input_source.type IN ('CAST','NATIVE APP','VIAPLUS') THEN 'APPS'
                      WHEN input_source.type = 'USB' THEN 'OTHER'
                      WHEN input_source.type = 'ANALOG TV' THEN 'SD TV'
                      WHEN input_source.type = 'DIGITAL ANTENNA' AND percent_hd_frames > 0.85 THEN 'HD TV'
                      WHEN input_source.type = 'DIGITAL ANTENNA'
                        AND percent_hd_frames > percent_sd_frames
                        AND percent_hd_frames > percent_other_frames 
                        AND percent_local_frames >= 0.99 THEN 'HD TV'
                      WHEN input_source.type = 'DIGITAL ANTENNA'
                        AND percent_sd_frames > percent_hd_frames
                        AND percent_sd_frames > percent_other_frames 
                        AND percent_local_frames >=0.99 THEN 'SD TV'
                      WHEN input_source.type = 'DIGITAL ANTENNA' 
                        AND percent_other_frames > percent_hd_frames
                        AND percent_other_frames > percent_sd_frames 
                        AND percent_local_frames >=0.99 THEN 'OTHER'
                      WHEN detection_rate > 0.2 AND percent_hd_frames > 0.85 THEN 'HD TV'
                      WHEN detection_rate > 0 
                        AND detection_rate <= 0.2 AND percent_hd_frames > percent_sd_frames
                        AND percent_hd_frames > percent_other_frames
                        AND input_device_prelim IN ('XFINITY','DirecTV','Spectrum','Dish','Verizon_Fios','Cox','Altice',
                                                  'Uverse','MediaCom','Frontier','TiVo','Cable One'
                                                 ) THEN 'HD TV'
                      WHEN detection_rate > 0 AND detection_rate <= 0.2 
                        AND percent_hd_frames > percent_sd_frames
                        AND percent_hd_frames > percent_other_frames
                        AND (input_device_prelim NOT IN ('XFINITY','DirecTV','Spectrum','Dish','Verizon_Fios','Cox','Altice',
                                              'Uverse','MediaCom','Frontier','TiVo','Cable One'
                                             ) OR input_device_prelim IS NULL) THEN 'OTT'
                      WHEN percent_sd_frames > percent_hd_frames AND percent_sd_frames > percent_other_frames THEN 'SD TV'
                      ELSE 'OTHER'
                  END AS category

        FROM {catalog_temp}.{db_name_temp}.input_stats_1 as s1
        JOIN detection.input_source ON input_source.input_source_id = s1.fk_input_source_id
        """)

    input_stats_2 = deltaHelpers.saveToDeltaTempTable(input_stats_2, "input_stats_2")

    input_stats_3 = spark.sql(f"""
          SELECT s2.*,
              CASE WHEN input_source.type IN ('CAST', 'NATIVE APP', 'VIAPLUS','USB') THEN input_source.type
                        WHEN input_source.type IN ('ANALOG TV','DIGITAL ANTENNA')
                          AND percent_local_frames >= .99
                          AND category IN ('SD TV','HD TV', 'OTHER') THEN 'OTA'
                        WHEN input_source.type = 'ANALOG TV' THEN input_source.type
                        WHEN input_source.type = 'DIGITAL ANTENNA' AND percent_hd_frames > 0.85 THEN 'HD DTV'
                        WHEN input_source.type = 'DIGITAL ANTENNA' AND percent_sd_frames > percent_hd_frames AND percent_sd_frames > percent_other_frames THEN 'SD DTV'
                        WHEN detection_rate > 0.2 AND percent_hd_frames > 0.85 THEN 'HD STB'
                        WHEN detection_rate > 0 AND detection_rate <= 0.2 AND percent_hd_frames > percent_sd_frames
                          AND percent_hd_frames > percent_other_frames
                          AND input_device_prelim IN ('XFINITY','DirecTV','Spectrum','Dish','Verizon_Fios','Cox','Altice',
                                                    'Uverse','MediaCom','Frontier','TiVo','Cable One'
                                                   ) THEN 'HD STB'
                        WHEN detection_rate > 0 AND detection_rate <= 0.2 AND percent_hd_frames > percent_sd_frames AND percent_hd_frames > percent_other_frames THEN 'OTT'
                        WHEN detection_rate = 0 AND percent_hd_frames > percent_sd_frames AND percent_hd_frames > percent_other_frames THEN 'UNINGESTED'
                        WHEN detection_rate > 0.2 AND percent_sd_frames > percent_hd_frames AND percent_sd_frames > percent_other_frames THEN 'SD STB'
          --               WHEN detection_rate <= 0.2 AND percent_sd_frames > percent_hd_frames AND percent_sd_frames > percent_other_frames THEN 'OTHER'
                        ELSE 'OTHER'
                    END AS subcategory
          FROM {catalog_temp}.{db_name_temp}.input_stats_2 s2
          JOIN detection.input_source ON input_source.input_source_id = s2.fk_input_source_id
          """)

    input_stats_3 = deltaHelpers.saveToDeltaTempTable(input_stats_3, "input_stats_3")

    input_stats_4 = spark.sql(f"""
          SELECT *,
             CASE
                     WHEN category = 'APPS' THEN NULL
                     WHEN subcategory = 'OTA' THEN subcategory
                     WHEN category = 'SD TV' AND input_device_prelim  IN ('Amazon_Fire','Chromecast','Roku','Apple_TV','PlayStation 4','Nintendo Switch','PlayStation 5','XBOX')  THEN NULL
                     WHEN category = 'HD TV' AND subcategory = 'HD DTV' AND input_device_prelim  IN ('Amazon_Fire','Chromecast','Roku','Apple_TV','PlayStation 4','Nintendo Switch','PlayStation 5','XBOX' )  THEN NULL
                     WHEN category = 'OTHER' AND subcategory = 'USB'  THEN NULL
                     WHEN input_device_prelim IS NOT NULL THEN input_device_prelim
                    --  WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND detection_rate>=.2 THEN NULL
                     ELSE NULL
                  END AS input_device
          FROM {catalog_temp}.{db_name_temp}.input_stats_3 as s3
          """)
    input_stats_4 = deltaHelpers.saveToDeltaTempTable(input_stats_4, "input_stats_4")

    spark.sql(f"""
          INSERT INTO {catalog}.{stage_schema}.tv_input_stats_firehose_stage
          SELECT fk_tvid
               , fk_input_source_id
               , ROW_NUMBER() OVER (PARTITION BY fk_tvid, next_week_start ORDER BY detected_duration_by_input_source DESC) AS input_number
               , category
               , subcategory
               , CAST(detection_rate AS DECIMAL(4,2)) AS detection_rate
               , CAST(percent_hd_frames AS DECIMAL(4,2)) AS percent_hd_frames
               , CAST(percent_sd_frames AS DECIMAL(4,2)) AS percent_sd_frames
               , CAST(percent_other_frames AS DECIMAL(4,2)) AS percent_other_frames        
               , total_duration_by_input_source AS total_duration
               , next_week_start AS create_timestamp
               , '2100-01-01 00:00:00'::timestamp AS next_create_timestamp
               , percent_local_frames
               , input_device
                , CASE WHEN input_device IS NULL OR input_device = 'OTA' THEN NULL
                     WHEN logo_cec IS NOT NULL THEN logo_cec
                     WHEN category IN ('SD TV','HD TV') AND detection_rate>=.2 
                        AND isp IN ('Spectrum','Time%20Warner%20Cable','Charter%20Communications','Comcast%20Cable',
                                    'Verizon%20Fios','Verizon%20Internet%20Services','Verizon%20Wireless',
                                    'Cox%20Communications','Suddenlink%20Communications','Optimum%20Online',
                                    'Mediacom%20Cable','Cable%20One') THEN 'isp'
                     ELSE NULL
                  END AS input_device_source
                , CASE WHEN input_device IN ('PlayStation 4','Nintendo Switch','PlayStation 5','XBOX') THEN 'Gaming'
                       WHEN input_device  IN ('Amazon_Fire','Chromecast','Roku','Apple_TV'  ) THEN 'OTT Device'
                       WHEN input_device IN ('XFINITY','DirecTV','Spectrum','Dish','Verizon_Fios','Cox','Altice','Uverse','MediaCom','Frontier','TiVo','Cable One' ) THEN 'MVPD'
                       WHEN input_device_prelim IS NULL AND category IN ('SD TV','HD TV') AND detection_rate>=.2 THEN 'OTT Device'
                       ELSE NULL
                  END AS input_device_type
        FROM {catalog_temp}.{db_name_temp}.input_stats_4
        WHERE rnum_final = 1;
          """)
    
    CTE_4weekinputstats = spark.sql(f"""
     SELECT fk_tvid
          , fk_input_source_id
          , input_number
          , category
          , subcategory
          , detection_rate
          , percent_hd_frames
          , percent_sd_frames
          , percent_other_frames
          , 0 AS percent_local_frames
          , total_duration
          , create_timestamp
          , input_device
          , input_device_source
          , input_device_type
          , ROW_NUMBER() OVER(PARTITION BY fk_tvid,fk_input_source_id ORDER BY create_timestamp DESC) RN
     FROM {catalog}.{final_schema}.tv_input_stats_firehose --4 weeks ago, 1 week ago
     WHERE create_timestamp BETWEEN DATE_ADD('{to_date}'::date, -28) AND DATE_ADD('{to_date}'::date, -7)
     """)

    CTE_4weekinputstats = deltaHelpers.saveToDeltaTempTable(CTE_4weekinputstats, "CTE_4weekinputstats")

    spark.sql(f"""
      UPDATE {catalog}.{final_schema}.tv_input_stats_firehose
      SET next_create_timestamp = '{to_date}'
      WHERE next_create_timestamp =  '2100-01-01 00:00:00';
    """)
   
    spark.sql(f"""
     INSERT INTO {catalog}.{final_schema}.tv_input_stats_firehose
     SELECT fk_tvid
          , fk_input_source_id
          , input_number
          , category
          , CASE WHEN lag_subcategory = 'OTA' AND percent_local_frames IS NULL THEN 'OTA' ELSE subcategory END AS subcategory
          , detection_rate
          , percent_hd_frames
          , percent_sd_frames
          , percent_other_frames
          , total_duration
          , create_timestamp
          , next_create_timestamp
          , percent_local_frames
          , CASE WHEN lag_subcategory = 'OTA' AND percent_local_frames IS NULL THEN 'OTA' ELSE input_device END AS input_device
          , CASE WHEN lag_subcategory = 'OTA' AND percent_local_frames IS NULL THEN NULL ELSE input_device_source END AS input_device_source
          , CASE WHEN lag_subcategory = 'OTA' AND percent_local_frames IS NULL THEN NULL ELSE input_device_type END AS input_device_type
          FROM
          (SELECT a.fk_tvid
               , a.fk_input_source_id
               , a.input_number
               , a.category
               , a.subcategory
               , b.subcategory AS lag_subcategory
               , a.detection_rate
               , a.percent_hd_frames
               , a.percent_sd_frames
               , a.percent_other_frames
               , a.percent_local_frames
               , a.total_duration
               , a.create_timestamp
               , a.next_create_timestamp
               , a.input_device
               , a.input_device_source
               , a.input_device_type
          FROM {catalog}.{stage_schema}.tv_input_stats_firehose_stage a
          LEFT JOIN {catalog_temp}.{db_name_temp}.CTE_4weekinputstats b
          ON a.fk_tvid = b.fk_tvid
          AND a.fk_input_source_id = b.fk_input_source_id
          AND b.RN=1)
          UNION
          SELECT fk_tvid
          , fk_input_source_id
          , input_number
          , category
          , subcategory
          , 0 AS detection_rate
          , 0 AS percent_hd_frames
          , 0 AS percent_sd_frames
          , 0 AS percent_bad_frames
          , 0 AS total_duration
          , '{to_date}'::timestamp AS create_timestamp
          , '2100-01-01 00:00:00'::timestamp AS next_create_timestamp
          , 0 AS percent_local_frames
          , input_device
          , input_device_source
          , input_device_type
          FROM {catalog_temp}.{db_name_temp}.CTE_4weekinputstats 4weekinputstats 
          WHERE RN = 1
          AND NOT EXISTS ( SELECT 1 FROM {catalog}.{stage_schema}.tv_input_stats_firehose_stage WHERE 4weekinputstats.fk_tvid = tv_input_stats_firehose_stage.fk_tvid
          AND 4weekinputstats.fk_input_source_id = tv_input_stats_firehose_stage.fk_input_source_id);
     """)